# EXPRESSÕES REGULARES E FUNÇÕES

Limpeza de dados + funções reutilizáveis

**objetivos**:

- Limpar caracteres
- Padronizar
- Verificar critérios



## Ler a base de clientes e fazer o diagnóstico inicial

In [1]:
from google.colab import files
uploaded = files.upload()

Saving clientes_sujos.csv to clientes_sujos.csv


In [2]:
import pandas as pd
import re

clientes = pd.read_csv(
    "clientes_sujos.csv",
    dtype={
        "telefone": "string",
        "cpf": "string",
        "email": "string"
    }
)

In [3]:
print("Tamanho da tabela: ", clientes.shape)

clientes.info()

print(clientes.isna().sum())

display(
    clientes[["nome", "telefone", "email", "cpf"]]
)

Tamanho da tabela:  (8, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cliente_id     8 non-null      int64 
 1   nome           8 non-null      object
 2   telefone       8 non-null      string
 3   email          8 non-null      string
 4   cpf            8 non-null      string
 5   data_cadastro  8 non-null      object
 6   observacao     8 non-null      object
dtypes: int64(1), object(3), string(3)
memory usage: 580.0+ bytes
cliente_id       0
nome             0
telefone         0
email            0
cpf              0
data_cadastro    0
observacao       0
dtype: int64


,nome,telefone,email,cpf
0,Ana Souza,(21) 98888-1111,ana.souza@email.com,123.456.789-10
1,Bruno Lima,21 97777 2222,bruno.lima@email,98765432100
2,Carla Dias,+55 (21) 96666-3333,carla.dias@empresa.com.br,111.222.333-44
3,Diego Rocha,(11)95555-4444,diego_rocha@email.com,22233344455
4,Eva Martins,21944445555,eva.martins@email.com,333.444.555-66
5,Felipe Nunes,sem telefone,felipe.nunes@email.com,444.555.666-77
6,Giovana Reis,(31) 93333-7777,giovana.reis@email.com,55566677788
7,Hugo Alves,31-92222-8888,hugo.alves@empresa,666.777.888-99


# Introdução regex com um único telefone



In [4]:
texto = "(21) 98888-1111"

somente_numeros = re.sub(r"\D", "", texto)
print(somente_numeros)

21988881111


In [6]:
cpf = "123.456.789-10"

cpf_limpo = re.sub(r"\D", "", cpf)
print(cpf_limpo)

12345678910


# Transformar a limpeza em uma função

In [15]:
def limpar_numero(texto):
  if pd.isna(texto):
    return ""

  texto = str(texto)

  return re.sub(r"\D", "", texto)

In [16]:
telefone = limpar_numero(texto="(21) 98888-1111")
print(telefone)

21988881111


In [17]:
print(limpar_numero("123.456.789"))
print(limpar_numero("sem telefone"))
print(limpar_numero("012.345.678-90"))

123456789

01234567890


In [18]:
clientes["telefone_limpo"] = clientes["telefone"].apply(limpar_numero)

clientes["cpf_limpo"] = clientes["cpf"].apply(limpar_numero)

In [19]:
display(
    clientes[
        ["nome", "telefone", "telefone_limpo", "cpf", "cpf_limpo"]
    ]
)

,nome,telefone,telefone_limpo,cpf,cpf_limpo
0,Ana Souza,(21) 98888-1111,21988881111,123.456.789-10,12345678910
1,Bruno Lima,21 97777 2222,21977772222,98765432100,98765432100
2,Carla Dias,+55 (21) 96666-3333,5521966663333,111.222.333-44,11122233344
3,Diego Rocha,(11)95555-4444,11955554444,22233344455,22233344455
4,Eva Martins,21944445555,21944445555,333.444.555-66,33344455566
5,Felipe Nunes,sem telefone,,444.555.666-77,44455566677
6,Giovana Reis,(31) 93333-7777,31933337777,55566677788,55566677788
7,Hugo Alves,31-92222-8888,31922228888,666.777.888-99,66677788899


# Introdução de lambda

para limpar os espaços dos emails.

In [21]:
emails_para_limpar = clientes["email"].fillna("")

clientes["email_limpo"] = emails_para_limpar.apply(
    lambda texto: str(texto).strip()
)

"ana.souza@email.com "
"ana.souza@email.com"

In [22]:
display(
    clientes[
        ["email", "email_limpo"]
    ]
)

,email,email_limpo
0,ana.souza@email.com,ana.souza@email.com
1,bruno.lima@email,bruno.lima@email
2,carla.dias@empresa.com.br,carla.dias@empresa.com.br
3,diego_rocha@email.com,diego_rocha@email.com
4,eva.martins@email.com,eva.martins@email.com
5,felipe.nunes@email.com,felipe.nunes@email.com
6,giovana.reis@email.com,giovana.reis@email.com
7,hugo.alves@empresa,hugo.alves@empresa


# Verificar o formato dos emails com uma função

In [23]:
def email_valido(email):
  if pd.isna(email):
    return False

  email = str(email).strip()

  padrao = r"^[\w\.-]+@[\w\.-]+\.\w+$"

  return bool(re.match(padrao, email))

In [28]:
print(email_valido("ana.souza@email.com"))
print(email_valido("bruno.lima@email"))
print(email_valido("carla.dias@empresa.com.br"))

True
False
True


In [26]:
clientes["email_formato_ok"] = clientes["email_limpo"].apply(email_valido)